# Get citing & cited opinions metadata

The "scotus.csv" file produced by clean_output.ipynb contains the citing and cited opinion cluster ids. The next step is to get the metadata and the raw citing opinions (including all sub types of opinions) for the citing opinions so that we can feed the opinion to the model to make prediction.

This notebook documents the steps I undertook to:
1. Use Django shell commands to extract the citing_opinion metadata & raw opinions & join to scotus.csv, this produces citing_joined df
2. Use Django shell commands to identify all cited_opinions for the ~9k citing opinions, get the metadata & citation depth for all cited_opinions & join to citing_joined df, this produces the final df

# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import json
import random
import os

# Load the target scotus data

In [2]:
scotus = pd.read_csv("citing/scotus.csv")
scotus = scotus[["cited_cluster_id", "citing_cluster_id"]]
scotus.head()

,cited_cluster_id,citing_cluster_id
0,84681,4877180
1,84681,214949
2,84681,4895709
3,84681,95391
4,84681,95822


In [3]:
citing_ids = scotus["citing_cluster_id"].unique()
len(citing_ids)

9402

In [4]:
cited_ids = scotus["cited_cluster_id"].unique()
len(cited_ids)

964

# Load some metadata that we already have

In [5]:
data = pd.read_csv("../experiments_501/data/output_dataset.csv")
data.head()

,citing_cluster_id,citing_decision_name,citing_url,citing_opinions,citing_filenames,cited_cluster_id,cited_decision_name,cited_url,cited_name_short,cited_name,cited_name_full,cited_citations,overruled,note,use_full_opinion,filename
0,91306,"Morgan v. United States,113 U.S. 476 (1885)",https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt'],88061,Texas v. White (1869),https://www.courtlistener.com/opinion/88061/te...,White,Texas v. White,Texas v. White Et Al.,"['74 U.S. 700', '19 L. Ed. 227', '7 Wall. 700'...",yes,NaN,0,0001.91306_cites_88061.txt
1,91306,"Morgan v. United States,113 U.S. 476 (1885)",https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt'],88994,Vermilye & Co. v. Adams Express Co. (1875),https://www.courtlistener.com/opinion/88994/ve...,NaN,Vermilye & Co. v. Adams Express Co.,Vermilye & Co. v. Adams Express Company,"['88 U.S. 138', '22 L. Ed. 609', '21 Wall. 138...",no,NaN,0,0002.91306_cites_88994.txt
2,91306,"Morgan v. United States,113 U.S. 476 (1885)",https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt'],87633,Murray v. Lardner (1865),https://www.courtlistener.com/opinion/87633/mu...,Murray,Murray v. Lardner,Murray v. Lardner,"['69 U.S. 110', '17 L. Ed. 857', '2 Wall. 110'...",no,NaN,0,0003.91306_cites_87633.txt
3,91306,"Morgan v. United States,113 U.S. 476 (1885)",https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt'],88240,Texas v. Hardenberg (1869),https://www.courtlistener.com/opinion/88240/te...,Hardenberg,Texas v. Hardenberg,Texas v. Hardenberg,"['77 U.S. 68', '19 L. Ed. 839', '10 Wall. 68',...",no,NaN,0,0004.91306_cites_88240.txt
4,91306,"Morgan v. United States,113 U.S. 476 (1885)",https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt'],88693,Huntington v. Texas (1873),https://www.courtlistener.com/opinion/88693/hu...,Huntington,Huntington v. Texas,Huntington v. Texas; Texas v. Huntington,"['83 U.S. 402', '21 L. Ed. 316', '16 Wall. 402...",no,NaN,0,0005.91306_cites_88693.txt


In [6]:
data.columns

Index(['citing_cluster_id', 'citing_decision_name', 'citing_url',
       'citing_opinions', 'citing_filenames', 'cited_cluster_id',
       'cited_decision_name', 'cited_url', 'cited_name_short', 'cited_name',
       'cited_name_full', 'cited_citations', 'overruled', 'note',
       'use_full_opinion', 'filename'],
      dtype='object')

In [7]:
citing_data = data[['citing_cluster_id', 'citing_url', 'citing_opinions', 'citing_filenames']].drop_duplicates()
len(citing_data)

133

In [8]:
cited_data = data[['cited_cluster_id', 'cited_decision_name', 'cited_url', 'cited_name_short', 'cited_name', 'cited_name_full', 'cited_citations']].drop_duplicates()
len(cited_data)

971

# Identify the citing_opinion cluster ids that we need to get the metadata & opinions for

In [9]:
get_citing_data = list(set(citing_ids) - set(citing_data["citing_cluster_id"].unique()))
len(get_citing_data)

9269

# Save the list of citing cluster_ids to a txt file to be used in the Django Shell

In [10]:
#with open('data/cluster_ids.txt', 'w') as f:
#    for number in get_citing_data:
#        f.write(f"{number}\n")

# Load the citing opinions results from Django Shell

The process_citing_opinions() function in script.py file in the data folder is the script used to retrieve the citing opinion metadata & opinion raw text from CLReplica, the script was run inside the Django Shell within CourtListener repo with access to CLReplica. The metadata are saved in citing_opinions.json file in the data folder and the opinion raw texts are saved in the raw_citing_opinions folder

In [11]:
with open('data/citing_opinions.json', 'r') as f:
    results = json.load(f)

In [12]:
records = []
for case_id, case_data in results.items():
    record = {'citing_cluster_id': int(case_id)}
    record.update({k: v for k, v in case_data.items()})
    
    opinion_filenames = [op['opinion_filename'] for op in case_data.get('opinion_data', [])]
    record['opinion_filenames'] = opinion_filenames
    
    records.append(record)

citing_df = pd.DataFrame(records)
citing_df.head()

,citing_cluster_id,case_law_url,case_name_short,case_name,case_name_full,citation_names,opinion_data,opinion_filenames
0,4877180,https://www.courtlistener.com/opinion/4877180/...,,Texas v. California,,[],"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt]
1,214949,https://www.courtlistener.com/opinion/214949/v...,Stewart,Virginia Office for Protection and Advocacy v....,,[],"[{'opinion_id': 214949, 'opinion_api': 'https:...",[214949_010combined.txt]
2,4895709,https://www.courtlistener.com/opinion/4895709/...,,PennEast Pipeline Co. v. New Jersey,,[],"[{'opinion_id': 4699488, 'opinion_api': 'https...",[4895709_010combined.txt]
3,95391,https://www.courtlistener.com/opinion/95391/il...,,"Illinois Central Railroad Company, Appt. v. Wi...",Illinois Central Railroad Company v. Adams; Il...,"[1901 U.S. LEXIS 1280, 45 L. Ed. 410, 21 S. Ct...","[{'opinion_id': 95391, 'opinion_api': 'https:/...",[95391_010combined.txt]
4,95822,https://www.courtlistener.com/opinion/95822/he...,Hennessy,Hennessy v. Richardson Drug Co.,Hennessy v. Richardson Drug Company,"[1903 U.S. LEXIS 1322, 47 L. Ed. 697, 23 S. Ct...","[{'opinion_id': 95822, 'opinion_api': 'https:/...",[95822_010combined.txt]


In [13]:
assert set(get_citing_data) == set(citing_df["citing_cluster_id"].unique())

## Clean up the new citing data so we can join the new and old citing metadata

In [14]:
citing_data.columns

Index(['citing_cluster_id', 'citing_url', 'citing_opinions',
       'citing_filenames'],
      dtype='object')

In [15]:
citing_data.head()

,citing_cluster_id,citing_url,citing_opinions,citing_filenames
0,91306,https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt']
8,92059,https://www.courtlistener.com/opinion/92059/in...,"[{'opinion_id': 9417465, 'opinion_api': 'https...","['92059_020lead.txt', '92059_030concurrence.tx..."
15,92291,https://www.courtlistener.com/opinion/92291/le...,"[{'opinion_id': 92291, 'opinion_api': 'https:/...",['92291_010combined.txt']
20,93311,https://www.courtlistener.com/opinion/93311/br...,"[{'opinion_id': 93311, 'opinion_api': 'https:/...",['93311_010combined.txt']
25,93904,https://www.courtlistener.com/opinion/93904/ro...,"[{'opinion_id': 93904, 'opinion_api': 'https:/...",['93904_010combined.txt']


In [16]:
citing_df.columns

Index(['citing_cluster_id', 'case_law_url', 'case_name_short', 'case_name',
       'case_name_full', 'citation_names', 'opinion_data',
       'opinion_filenames'],
      dtype='object')

In [17]:
citing_df.rename(columns={"case_law_url": "citing_url",
                  "opinion_data": "citing_opinions",
                  "opinion_filenames": "citing_filenames"}, inplace=True)

citing_df = citing_df[['citing_cluster_id', 'citing_url', 'citing_opinions', 'citing_filenames']]

In [18]:
citing_df.head()

,citing_cluster_id,citing_url,citing_opinions,citing_filenames
0,4877180,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt]
1,214949,https://www.courtlistener.com/opinion/214949/v...,"[{'opinion_id': 214949, 'opinion_api': 'https:...",[214949_010combined.txt]
2,4895709,https://www.courtlistener.com/opinion/4895709/...,"[{'opinion_id': 4699488, 'opinion_api': 'https...",[4895709_010combined.txt]
3,95391,https://www.courtlistener.com/opinion/95391/il...,"[{'opinion_id': 95391, 'opinion_api': 'https:/...",[95391_010combined.txt]
4,95822,https://www.courtlistener.com/opinion/95822/he...,"[{'opinion_id': 95822, 'opinion_api': 'https:/...",[95822_010combined.txt]


# Create full citing data metadata

In [19]:
citing_metadata = pd.concat([citing_data.reset_index(drop=True), citing_df.reset_index(drop=True)])
len(citing_metadata)

9402

In [20]:
assert set(citing_metadata["citing_cluster_id"].unique()) == set(scotus["citing_cluster_id"].unique())

# Save the list of citing cluster_ids to a txt file to be used in the Django Shell

In [21]:
#with open('data/cluster_ids.txt', 'w') as f:
#    for number in citing_ids:
#        f.write(f"{number}\n")

# Load the cited opinions depth from Django Shell

The get_cited_opinions() function in script.py file in the data folder is the script used to retrieve the cited opinion metadata & the citation depth from CLReplica, the script was run inside the Django Shell within CourtListener repo with access to CLReplica. The metadata are saved in cited_clusters_metadata.json file in the data folder and the citation depths are saved in the cited_opinions.json file.

In [22]:
with open('data/cited_opinions.json', 'r') as f:
    results = json.load(f)

In [23]:
rows = []
for citing_id, cited_dict in results.items():
    for cited_id, depth in cited_dict.items():
        rows.append({
            "citing_cluster_id": int(citing_id),
            "cited_cluster_id": int(cited_id),
            "citation_depth": int(depth)
        })

cited_depth = pd.DataFrame(rows)
cited_depth.head()

,citing_cluster_id,cited_cluster_id,citation_depth
0,4877180,108291,5
1,4877180,95159,4
2,4877180,109452,4
3,4877180,108523,3
4,4877180,110972,3


In [24]:
set(citing_ids) - set(cited_depth["citing_cluster_id"].unique())

{np.int64(87754), np.int64(94419), np.int64(1087856), np.int64(2402836)}

4 citing opinions are missing, upon investigation, I believe it's due to the on-going process of get_citation script ran by the case law team, which causes some discrepancies in the data. It appears the "authorities" page for these citing_cluster_ids are blank, therefore, the cited_cluster_ids were not retrieved as part of the script. For these records, I set the citation_depth as 1 for now.

In [25]:
missed_cited_depth = scotus[scotus["citing_cluster_id"].isin([87754, 94419, 1087856, 2402836])]
missed_cited_depth["citation_depth"] = 1
missed_cited_depth

/var/folders/rx/3t2jtk8j69j_db_cydq_v6wc0000gn/T/ipykernel_95102/1185439706.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missed_cited_depth["citation_depth"] = 1


,cited_cluster_id,citing_cluster_id,citation_depth
5162,93379,94419,1
5261,93638,2402836,1
27618,106545,1087856,1
42599,1528394,87754,1


## Combine the cited_depth and the missed_cited_depth to create depth dataframe

In [26]:
depth = pd.concat([cited_depth.reset_index(drop=True), missed_cited_depth.reset_index(drop=True)])
len(depth)

261218

## Confirm the same citing_cluster_ids are in scotus and depth

In [27]:
assert set(scotus["citing_cluster_id"].unique()) == set(depth["citing_cluster_id"].unique())

## Check to confirm the citing-cited pairs in scotus exist in depth

As expected, the citing-cited pairs from scotus are fewer than that from depth, since depth contains all the cited references (authorities) for the citings, whereas scotus only contains the scotus citings for each cited (cited-by). Theoretically, all citing-cited pairs from scotus should appear in depth. However, I found that 596 of these pairs are missing from depth. I sampled a few to investigate and noted that this is due to data issue where the cited_cluster_ids change after find_citations rerun, such that when there are duplicated cited opinions, the linkage may change, hence, the cited_cluster_id changes. 

For example, in scotus, I see 100052 cites 93379, but 93379 is missing from depth. 93379 is 145 U.S. 263, but so is 8178682. So for the same citing-cited relationship, scotus shows 100052 cites 93379 whereas depth shows 10052 cites 8178682.

This is troublesome as I wanted to build a dataset which contains the entire subsequent citation history for all cited_cluster_ids in scotus, for these 596 cited_cluster_ids, this history is no longer present without manual reconciliation for duplication. 

To keep this moving forward, I decided to discard these cited_cluster_ids from the list of cited_cluster_ids for which we will identify the entire subsequent citation history. The 596 citing-cited pairs contains 85 cited_cluster_ids to discard, leaving 879 cited_cluster_ids for analysis.

In [28]:
scotus["citing_cited"] = scotus['citing_cluster_id'].astype(str) + '-' + scotus['cited_cluster_id'].astype(str)
depth["citing_cited"] = depth['citing_cluster_id'].astype(str) + '-' + depth['cited_cluster_id'].astype(str)

len(set(scotus["citing_cited"]) - set(depth["citing_cited"]))

596

In [29]:
set(scotus["citing_cited"]) - set(depth["citing_cited"])

{'100018-99906',
 '100052-93379',
 '100136-99906',
 '100155-1578327',
 '100180-99906',
 '100221-99906',
 '100222-93379',
 '100222-99906',
 '100232-1292673',
 '100240-84681',
 '100321-99906',
 '100380-99906',
 '100385-93638',
 '100475-88890',
 '100535-99906',
 '100635-98391',
 '100708-98337',
 '100771-94416',
 '100801-99906',
 '100821-99906',
 '100861-99906',
 '100953-1292673',
 '101011-100694',
 '101059-1292673',
 '101106-1292673',
 '101118-98094',
 '101119-97451',
 '101171-100916',
 '101259-99906',
 '101272-100916',
 '101291-100916',
 '101300-100916',
 '101365-101364',
 '101376-101295',
 '101413-97451',
 '101436-99906',
 '101499-100916',
 '101558-95204',
 '101573-99906',
 '101574-1578327',
 '101631-99906',
 '101632-100916',
 '101642-99906',
 '101731-100916',
 '101768-1292673',
 '101777-99906',
 '101797-99906',
 '101816-90041',
 '101832-93379',
 '101894-100916',
 '101961-101894',
 '101967-101915',
 '102024-100063',
 '102038-100916',
 '102038-101632',
 '102038-101894',
 '102041-1292673'

In [30]:
depth[depth["citing_cited"] == '100052-8178682']

,citing_cluster_id,cited_cluster_id,citation_depth,citing_cited
90098,100052,8178682,1,100052-8178682


In [31]:
discard = scotus[scotus["citing_cited"].isin(list(set(scotus["citing_cited"]) - set(depth["citing_cited"])))]["cited_cluster_id"].unique()
len(discard)

85

In [32]:
keep = list(set(cited_ids) - set(discard))
len(keep)

879

# Load the cited opinions metadata from Django Shell

In [33]:
with open('data/cited_clusters_metadata.json', 'r') as f:
    results = json.load(f)

In [34]:
cited_metadata = pd.DataFrame.from_dict(results, orient='index')
cited_metadata = cited_metadata.reset_index().rename(columns={'index': 'cited_cluster_id'})
cited_metadata.head()

,cited_cluster_id,cited_url,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,108291,https://www.courtlistener.com/opinion/108291/o...,Wyandotte Chemicals,Ohio v. Wyandotte Chemicals Corp.,OHIO v. WYANDOTTE CHEMICALS CORP. Et Al.,"[2 ERC (BNA) 1331, 1 Envtl. L. Rep. (Envtl. La..."
1,95159,https://www.courtlistener.com/opinion/95159/lo...,,Louisiana v. Texas,Louisiana v. Texas,"[1900 U.S. LEXIS 1715, 44 L. Ed. 347, 20 S. Ct..."
2,109452,https://www.courtlistener.com/opinion/109452/a...,AZ v. NM,Arizona v. New Mexico,Arizona v. New Mexico,"[1976 U.S. LEXIS 117, 425 U.S. 794, 96 S. Ct. ..."
3,108523,https://www.courtlistener.com/opinion/108523/i...,Milwaukee,Illinois v. City of Milwaukee,"ILLINOIS v. CITY OF MILWAUKEE, WISCONSIN, Et Al.","[4 ERC (BNA) 1001, 2 Envtl. L. Rep. (Envtl. La..."
4,110972,https://www.courtlistener.com/opinion/110972/t...,TX v. NM,Texas v. New Mexico,Texas v. New Mexico,"[51 U.S.L.W. 4805, 1983 U.S. LEXIS 67, 462 U.S..."


In [35]:
assert set(map(int, depth["cited_cluster_id"].unique())) == set(map(int, cited_metadata["cited_cluster_id"].unique()))

# Join the metadatas to produce final df

In [36]:
int_cols = ["citing_cluster_id", "cited_cluster_id", "citation_depth"]
depth[int_cols] = depth[int_cols].astype(int)
depth = depth.reset_index(drop=True)
depth.head()

,citing_cluster_id,cited_cluster_id,citation_depth,citing_cited
0,4877180,108291,5,4877180-108291
1,4877180,95159,4,4877180-95159
2,4877180,109452,4,4877180-109452
3,4877180,108523,3,4877180-108523
4,4877180,110972,3,4877180-110972


In [37]:
int_cols = ["citing_cluster_id"]
citing_metadata[int_cols] = citing_metadata[int_cols].astype(int)
citing_metadata = citing_metadata.reset_index(drop=True)
citing_metadata.head()

,citing_cluster_id,citing_url,citing_opinions,citing_filenames
0,91306,https://www.courtlistener.com/opinion/91306/mo...,"[{'opinion_id': 91306, 'opinion_api': 'https:/...",['91306_010combined.txt']
1,92059,https://www.courtlistener.com/opinion/92059/in...,"[{'opinion_id': 9417465, 'opinion_api': 'https...","['92059_020lead.txt', '92059_030concurrence.tx..."
2,92291,https://www.courtlistener.com/opinion/92291/le...,"[{'opinion_id': 92291, 'opinion_api': 'https:/...",['92291_010combined.txt']
3,93311,https://www.courtlistener.com/opinion/93311/br...,"[{'opinion_id': 93311, 'opinion_api': 'https:/...",['93311_010combined.txt']
4,93904,https://www.courtlistener.com/opinion/93904/ro...,"[{'opinion_id': 93904, 'opinion_api': 'https:/...",['93904_010combined.txt']


In [38]:
int_cols = ["cited_cluster_id"]
cited_metadata[int_cols] = cited_metadata[int_cols].astype(int)
cited_metadata = cited_metadata.reset_index(drop=True)
cited_metadata.head()

,cited_cluster_id,cited_url,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,108291,https://www.courtlistener.com/opinion/108291/o...,Wyandotte Chemicals,Ohio v. Wyandotte Chemicals Corp.,OHIO v. WYANDOTTE CHEMICALS CORP. Et Al.,"[2 ERC (BNA) 1331, 1 Envtl. L. Rep. (Envtl. La..."
1,95159,https://www.courtlistener.com/opinion/95159/lo...,,Louisiana v. Texas,Louisiana v. Texas,"[1900 U.S. LEXIS 1715, 44 L. Ed. 347, 20 S. Ct..."
2,109452,https://www.courtlistener.com/opinion/109452/a...,AZ v. NM,Arizona v. New Mexico,Arizona v. New Mexico,"[1976 U.S. LEXIS 117, 425 U.S. 794, 96 S. Ct. ..."
3,108523,https://www.courtlistener.com/opinion/108523/i...,Milwaukee,Illinois v. City of Milwaukee,"ILLINOIS v. CITY OF MILWAUKEE, WISCONSIN, Et Al.","[4 ERC (BNA) 1001, 2 Envtl. L. Rep. (Envtl. La..."
4,110972,https://www.courtlistener.com/opinion/110972/t...,TX v. NM,Texas v. New Mexico,Texas v. New Mexico,"[51 U.S.L.W. 4805, 1983 U.S. LEXIS 67, 462 U.S..."


In [39]:
depth_citing = depth.merge(citing_metadata, how="left", on="citing_cluster_id")
depth_cited = depth_citing.merge(cited_metadata, how="left", on="cited_cluster_id")

In [40]:
depth_cited.head()

,citing_cluster_id,cited_cluster_id,citation_depth,citing_cited,citing_url,citing_opinions,citing_filenames,cited_url,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,4877180,108291,5,4877180-108291,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/108291/o...,Wyandotte Chemicals,Ohio v. Wyandotte Chemicals Corp.,OHIO v. WYANDOTTE CHEMICALS CORP. Et Al.,"[2 ERC (BNA) 1331, 1 Envtl. L. Rep. (Envtl. La..."
1,4877180,95159,4,4877180-95159,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/95159/lo...,,Louisiana v. Texas,Louisiana v. Texas,"[1900 U.S. LEXIS 1715, 44 L. Ed. 347, 20 S. Ct..."
2,4877180,109452,4,4877180-109452,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/109452/a...,AZ v. NM,Arizona v. New Mexico,Arizona v. New Mexico,"[1976 U.S. LEXIS 117, 425 U.S. 794, 96 S. Ct. ..."
3,4877180,108523,3,4877180-108523,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/108523/i...,Milwaukee,Illinois v. City of Milwaukee,"ILLINOIS v. CITY OF MILWAUKEE, WISCONSIN, Et Al.","[4 ERC (BNA) 1001, 2 Envtl. L. Rep. (Envtl. La..."
4,4877180,110972,3,4877180-110972,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/110972/t...,TX v. NM,Texas v. New Mexico,Texas v. New Mexico,"[51 U.S.L.W. 4805, 1983 U.S. LEXIS 67, 462 U.S..."


In [41]:
assert depth_cited.isna().any(axis=1).sum() == 0

In [42]:
assert len(depth_cited) == len(depth)

# Tag the cited_cluster_id

With this dataset, we'll be able to produce the final (most recent & most severe) treatment for 879 cited opinions. However, since we will have to read through all subsequent citing opinions that cited these 879 opinion anyways, we might as well produce the treatment for all cited opinions within our citing opinions.

We should still tag the target 879 cited opinions.

In [43]:
depth_cited.loc[depth_cited["cited_cluster_id"].isin(keep), "cited_target"] = 1
depth_cited.loc[~depth_cited["cited_cluster_id"].isin(keep), "cited_target"] = 0

In [44]:
len(depth_cited)

261218

In [45]:
depth_cited.columns

Index(['citing_cluster_id', 'cited_cluster_id', 'citation_depth',
       'citing_cited', 'citing_url', 'citing_opinions', 'citing_filenames',
       'cited_url', 'cited_case_name_short', 'cited_case_name',
       'cited_case_name_full', 'cited_citations', 'cited_target'],
      dtype='object')

In [46]:
eda_cols = ['citing_cluster_id', 'cited_cluster_id', 'citation_depth', 'citing_cited', 'cited_target']

for col in eda_cols:
    print("----------")
    print(depth_cited[col].nunique())
    display(depth_cited[col].value_counts())

----------
9402


citing_cluster_id
106249     294
108329     254
106881     250
104616     231
101864     230
          ... 
9102853      1
9102854      1
9103280      1
9103279      1
87754        1
Name: count, Length: 9402, dtype: int64

----------
75081


cited_cluster_id
96405      624
85272      373
107252     240
84759      239
91573      238
          ... 
8785759      1
8760672      1
6719212      1
6910678      1
87430        1
Name: count, Length: 75081, dtype: int64

----------
222


citation_depth
2      101355
1       76898
4       32141
6        9902
3        9753
        ...  
205         1
185         1
238         1
220         1
210         1
Name: count, Length: 222, dtype: int64

----------
261218


citing_cited
4877180-108291    1
131149-2620886    1
131149-107701     1
131149-109462     1
131149-106142     1
                 ..
95174-6441199     1
105412-93107      1
105412-105326     1
105412-99954      1
87754-1528394     1
Name: count, Length: 261218, dtype: int64

----------
2


cited_target
0.0    233642
1.0     27576
Name: count, dtype: int64

# Save the df for future use

In [47]:
depth_cited.to_json("data/scotus_citing_cited.json")

# Sample from the final df to produce a small sample set for expert annotation

In [48]:
random.seed(42)
keep_sampled = random.sample(keep, 40)
len(keep_sampled)

40

In [49]:
cited_sampled = depth_cited[depth_cited["cited_cluster_id"].isin(keep_sampled)]
citing_sampled = cited_sampled["citing_cluster_id"].unique()

In [50]:
depth_sampled = depth_cited[depth_cited["citing_cluster_id"].isin(citing_sampled)]
len(depth_sampled)

41514

In [51]:
depth_sampled["citing_cited"].nunique()

41514

In [52]:
depth_sampled["citing_cluster_id"].nunique()

965

Assume each citing opinion takes an expert 15 minutes (on average) to annotate, with 20 experts at 10 hours per week and at least 3 experts for each opinion to produce triple-labels to reduce subjective bias, we should be done with the ~1000 citing opinions in ~4 weeks. This will produce more than 40k unique citing-cited pair treatments, and we will be able to produce the final authoritative treatments for 40 cited scotus opinions (note that since these are scotus opinions, only scotus citing opinions are considered for the final authoritative treatments).

In [53]:
depth_sampled[depth_sampled["cited_cluster_id"].isin(keep_sampled)][["citing_cluster_id", "cited_cluster_id"]].value_counts("cited_cluster_id")

cited_cluster_id
103347     157
104504     122
109097      83
99296       76
108184      47
109016      47
95346       42
111904      40
96764       39
96878       35
101911      34
93803       31
100188      29
106777      28
101568      27
96689       27
112205      26
112291      24
112052      18
108760      17
104918      17
110927      17
102412      15
106558      14
102602      12
105789      11
107449      10
109961      10
96107        9
108193       8
106050       8
8180960      8
106562       4
98481        4
100830       3
2305304      2
8928994      2
1280169      1
423986       1
88693        1
Name: count, dtype: int64

In [54]:
for col in eda_cols:
    print("----------")
    print(depth_sampled[col].nunique())
    display(depth_sampled[col].value_counts())

----------
965


citing_cluster_id
108329     254
101864     230
106366     227
106107     200
106267     186
          ... 
2395615      1
9086798      1
9086799      1
9091268      1
9091269      1
Name: count, Length: 965, dtype: int64

----------
16753


cited_cluster_id
103347     157
104504     122
109097      83
103243      77
99296       76
          ... 
86185        1
85584        1
94050        1
5700900      1
6680988      1
Name: count, Length: 16753, dtype: int64

----------
145


citation_depth
2      18490
1       8287
4       5840
6       1929
3       1417
       ...  
224        1
110        1
135        1
342        1
119        1
Name: count, Length: 145, dtype: int64

----------
41514


citing_cited
117880-110033     1
110559-107730     1
110559-5684296    1
110559-108893     1
110559-109714     1
                 ..
107487-103050     1
107487-106285     1
107487-107439     1
107487-106862     1
118219-6680988    1
Name: count, Length: 41514, dtype: int64

----------
2


cited_target
0.0    35073
1.0     6441
Name: count, dtype: int64

## Check how many in the sampled dataset are overruled

The ratio is in-line with expectation.

In [55]:
data[data["cited_cluster_id"].isin(keep_sampled)].value_counts("overruled")

overruled
no     36
yes     8
Name: count, dtype: int64

In [56]:
data[data["cited_cluster_id"].isin(keep_sampled)].value_counts("citing_cluster_id")

citing_cluster_id
98917     2
101913    2
107473    2
112640    2
112608    2
112258    2
111404    1
109252    1
109450    1
110212    1
110325    1
110719    1
91306     1
111555    1
108730    1
112739    1
112906    1
117958    1
118011    1
111940    1
107973    1
108362    1
103869    1
99004     1
99901     1
103442    1
103493    1
103522    1
103736    1
103870    1
107919    1
103962    1
104610    1
105319    1
106235    1
107252    1
107689    1
118317    1
Name: count, dtype: int64

## Save for expert annotation

In [57]:
depth_sampled.to_json("data/scotus_citing_cited_sampled.json")

In [58]:
annotation = depth_sampled[['citing_cluster_id', 'citing_url', 'cited_cluster_id', 'cited_url', 'cited_case_name_short', 'cited_case_name', 'cited_citations']]
annotation.head()

,citing_cluster_id,citing_url,cited_cluster_id,cited_url,cited_case_name_short,cited_case_name,cited_citations
764,117880,https://www.courtlistener.com/opinion/117880/h...,110033,https://www.courtlistener.com/opinion/110033/l...,,"Lake Country Estates, Inc. v. Tahoe Regional P...","[1979 U.S. LEXIS 68, 440 U.S. 391, 99 S. Ct. 1..."
765,117880,https://www.courtlistener.com/opinion/117880/h...,112423,https://www.courtlistener.com/opinion/112423/p...,Feeney,Port Authority Trans-Hudson Corp. v. Feeney,"[58 U.S.L.W. 4536, 1990 U.S. LEXIS 2294, 495 U..."
766,117880,https://www.courtlistener.com/opinion/117880/h...,522174,https://www.courtlistener.com/opinion/522174/p...,,Patrick Feeney v. Port Authority Trans-Hudson ...,"[1989 WL 41758, 1989 U.S. App. LEXIS 5959, 873..."
767,117880,https://www.courtlistener.com/opinion/117880/h...,463291,https://www.courtlistener.com/opinion/463291/a...,,Alfred Morris v. Washington Metropolitan Area ...,"[39 Empl. Prac. Dec. (CCH) 35,824, 1986 U.S. A..."
768,117880,https://www.courtlistener.com/opinion/117880/h...,1455885,https://www.courtlistener.com/opinion/1455885/...,Hess,Hess v. Port Authority Trans-Hudson Corp.(PATH),"[1992 WL 395860, 1992 U.S. Dist. LEXIS 20380, ..."


# Assign to experts based on their availability

In [59]:
more_than_ten = [] # volunteer names removed before pushing to publish repo
five_to_ten = [] # volunteer names removed before pushing to publish repo
less_than_five = [] # volunteer names removed before pushing to publish repo

In [60]:
capacity_map = {}

more_than_ten_max = {name: 12*60*4 for name in more_than_ten} # ~12hrs/week for 4 weeks
five_to_ten_max = {name: 7.5*60*3.8 for name in five_to_ten} # ~7.5hrs/week for 3.8 weeks
less_than_five_max = {name: 2.5*60*4 for name in less_than_five} # ~2.5hrs/week for 4 weeks

capacity_map.update(more_than_ten_max)
capacity_map.update(five_to_ten_max)
capacity_map.update(less_than_five_max)

In [61]:
assignment = annotation.groupby(["citing_cluster_id"]).size().reset_index(name="num_authorities") #assume each authority takes 1 minute to annotate
assignment.head()

,citing_cluster_id,num_authorities
0,1741,146
1,91306,10
2,93803,67
3,93850,31
4,94117,28


In [62]:
for i, row in assignment.iterrows():
    for name, available in capacity_map.items():
        if available >= row['num_authorities']:
            assignment.at[i, 'expert'] = name
            capacity_map[name] -= row['num_authorities']
            break

In [63]:
assert len(assignment[assignment["expert"].isna()]) == 0

In [64]:
assert set(assignment["expert"].unique()) == set(more_than_ten + five_to_ten + less_than_five)

In [65]:
double_check = assignment.groupby('expert')['num_authorities'].sum().reset_index()

for expert, max_mins in more_than_ten_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

for expert, max_mins in five_to_ten_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

for expert, max_mins in less_than_five_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

In [66]:
assignment_dict = dict(zip(assignment['citing_cluster_id'], assignment['expert']))

In [67]:
for citing_cluster_id in annotation["citing_cluster_id"].unique():
    annotate = annotation[annotation["citing_cluster_id"] == citing_cluster_id].copy()
    expert = assignment_dict[citing_cluster_id]
    annotate["expert"] = expert
    annotate["expert_label"] = None
    num_cited = annotate["cited_cluster_id"].nunique()

    expert_folder = f"data/annotations/{expert}"
    os.makedirs(expert_folder, exist_ok=True)
    
    annotate.to_csv(f"{expert_folder}/{num_cited}_{citing_cluster_id}.csv", index=False)